# Gemini ANY Mode: Enum Complexity Causes Opaque `INVALID_ARGUMENT`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/radishbuild/field-notes/blob/main/gemini-any-mode-enum-bug/repro_minimal.ipynb)

## What this reproduces

When using Gemini's `FunctionCallingConfig(mode="ANY")`, the API returns an opaque `INVALID_ARGUMENT` error when tool schemas contain many enum fields with multi-token values. **AUTO mode with identical schemas works fine.**

This notebook demonstrates that the failure depends on **enum string content**, not schema structure. I generate identical schemas (same number of tools, enum fields, and values per enum) and only vary the enum strings:

- **1-word values** (`"action"`, `"pending"`) → **PASS**
- **2-word values** (`"action_required"`, `"deploy_failed"`) → **FAIL**
- **3-word values** (`"action_required_review"`) → **FAIL**
- **4-word values** (`"action_required_review_pending"`) → **FAIL**

## Hypothesis

I suspect ANY mode hits some opaque internal budget limit — likely related to token count or schema complexity — that is undocumented and produces zero diagnostic information. Multi-token compound strings (like `action_required`) consume more of this budget than single-token values, causing the request to be rejected with a bare `INVALID_ARGUMENT` error once the budget is exceeded.

In [ ]:
!pip install -q google-genai

## API Key Setup

Go to **Colab menu → 🔑 Secrets** (key icon in left sidebar), add a secret named `GEMINI_API_KEY` with your [Gemini API key](https://aistudio.google.com/apikey), and enable notebook access.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
print("Client ready ✓")

## Configuration

Adjust these parameters to explore the budget boundary. Default config (20 × 4 × 3 = 240 total enum values) reliably shows 1-word PASS / 2-word FAIL.

In [ ]:
# Editable parameters
N_TOOLS = 20           # Number of tools
ENUMS_PER_TOOL = 4     # Enum fields per tool
VALUES_PER_ENUM = 3    # Values per enum field
MODEL = "gemini-3-flash-preview"

total_values = N_TOOLS * ENUMS_PER_TOOL * VALUES_PER_ENUM
print(f"Config: {N_TOOLS} tools × {ENUMS_PER_TOOL} enums/tool × {VALUES_PER_ENUM} values/enum = {total_values} total enum values")

## Word List & Pool Generation

I generate pools of enum values with 1, 2, 3, and 4 words joined by underscores. A seeded RNG picks random word combinations to avoid prefix sharing.

In [ ]:
import random
import json

WORDS = [
    "action", "required", "pending", "completed", "triggered", "initiated",
    "exceeded", "deprecated", "configured", "terminated", "dispatched",
    "exhausted", "validated", "established", "authorized", "recommended",
    "accumulated", "interrupted", "compromised", "remediated", "propagated",
    "orchestrated", "quarantined", "synchronized", "throttled", "replicated",
    "invalidated", "negotiated", "escalated", "provisioned", "decommissioned",
    "encapsulated", "materialized", "acknowledged", "insufficient", "misconfigured",
    "unresponsive", "preemption", "degradation", "fragmentation", "reconciliation",
    "serialization", "vulnerability", "compatibility", "idempotent", "transparency",
    "observability", "remediation", "notification", "classification",
]


def generate_pool(n_words, min_size=600):
    """Generate a pool of unique enum values with n_words words joined by underscores."""
    rng = random.Random(42)  # deterministic

    if n_words == 1:
        pool = []
        for i in range(min_size):
            pool.append(f"{WORDS[i % len(WORDS)]}{i // len(WORDS) if i >= len(WORDS) else ''}")
        return pool[:min_size]

    seen = set()
    pool = []
    while len(pool) < min_size:
        combo = tuple(rng.sample(WORDS, n_words))
        if combo not in seen:
            seen.add(combo)
            pool.append("_".join(combo))
    return pool


POOLS = {n: generate_pool(n) for n in [1, 2, 3, 4]}

for n in [1, 2, 3, 4]:
    print(f"{n}-word sample: {POOLS[n][0]}")

In [ ]:
def make_tools(pool, n_tools, enums_per_tool, values_per_enum):
    """Generate n_tools tools, each with enums_per_tool enum fields."""
    needed = n_tools * enums_per_tool * values_per_enum
    assert needed <= len(pool), f"Need {needed} values but pool has {len(pool)}"

    idx = 0
    tools = []
    for i in range(n_tools):
        props = {
            "target": {"type": "string"},
            "count": {"type": "integer"},
            "enabled": {"type": "boolean"},
        }
        for j in range(enums_per_tool):
            values = pool[idx:idx + values_per_enum]
            idx += values_per_enum
            props[f"field_{j}"] = {"type": "string", "enum": values}

        tools.append({
            "name": f"tool_{i}",
            "description": f"Tool number {i} for testing",
            "parametersJsonSchema": {
                "type": "object",
                "properties": props,
                "required": ["target", "field_0"],
            },
        })
    return tools

print(f"make_tools ready ✓")

## Run Tests: 1-word through 4-word enum values

All 4 tests use **identical schema structure** (same number of tools, fields, enum count). Only the enum string content differs.

In [ ]:
def try_mode(client, fds, mode):
    """Try a generate_content call with the given mode. Returns (success, error)."""
    try:
        client.models.generate_content(
            model=MODEL,
            contents="hello",
            config=types.GenerateContentConfig(
                temperature=0,
                tools=[types.Tool(function_declarations=fds)],
                tool_config=types.ToolConfig(
                    function_calling_config=types.FunctionCallingConfig(mode=mode),
                ),
            ),
        )
        return True, None
    except Exception as e:
        return False, e.message if hasattr(e, "message") else str(e)


print(f"{'Words':>6} | {'Sample Value':<45} | Result")
print("-" * 70)

results = {}
all_tools = {}

for n_words in [1, 2, 3, 4]:
    pool = POOLS[n_words]
    tools = make_tools(pool, N_TOOLS, ENUMS_PER_TOOL, VALUES_PER_ENUM)
    fds = [types.FunctionDeclaration(**t) for t in tools]

    ok, err = try_mode(client, fds, "ANY")
    tag = "PASS ✓" if ok else "FAIL ✗"
    print(f"{n_words:>6} | {pool[0]:<45} | {tag}")
    if err:
        print(f"         Error: {err[:120]}")

    results[f"{n_words}_word"] = "PASS" if ok else "FAIL"
    all_tools[f"{n_words}_word"] = tools

## ANY vs AUTO Comparison

This cell takes the 2-word schema (which fails in ANY mode) and shows it works fine in AUTO mode. Same schema, different mode.

In [ ]:
# Use the 2-word tools that fail in ANY mode
failing_tools = all_tools["2_word"]
fds = [types.FunctionDeclaration(**t) for t in failing_tools]

print(f"Schema: {N_TOOLS} tools, {ENUMS_PER_TOOL} enum fields each, {VALUES_PER_ENUM} 2-word values per enum")
print(f"Total enum values: {total_values}")
print()

for mode in ["ANY", "AUTO"]:
    ok, err = try_mode(client, fds, mode)
    tag = "PASS ✓" if ok else "FAIL ✗"
    print(f"  mode={mode:<4}  {tag}")
    if err:
        print(f"           Error: {err[:120]}")

print()
print("Same schema. Same tools. Same enums. Only the mode differs.")

## Dump Schemas to JSON

Save the generated function declarations for manual inspection.

In [ ]:
dump = {
    "config": {
        "tools": N_TOOLS,
        "enums_per_tool": ENUMS_PER_TOOL,
        "values_per_enum": VALUES_PER_ENUM,
        "total_values": total_values,
    },
    "results": results,
    "tools_by_word_count": all_tools,
}

dump_json = json.dumps(dump, indent=2)
print(f"Dumped {len(dump_json)} bytes ({len(all_tools)} word-count variants, {N_TOOLS} tools each)")
print(f"Results: {results}")

# Uncomment to save to file:
# with open("repro_minimal_fds.json", "w") as f:
#     f.write(dump_json)